# Zemli: train the keera dosakaya (cucumber) disease model

**Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

Main data source: the *Plant Leaf Freshness and Disease Detection* dataset from Bangladesh vegetable fields (Hasan et al., 2024, CC-BY-4.0), hosted on Hugging Face, so it downloads with no login. Photos were taken with smartphones on a black background, so expect lower accuracy on real net-shed photos until pilot photos are added.

Optional: add a second dataset you downloaded yourself (step 3).

In [ ]:
!nvidia-smi -L
!pip -q install onnx datasets

## 1. Upload `zemli-prototype.zip`

In [ ]:
from google.colab import files
import glob, os, subprocess
up = files.upload()
# works whatever the file is called, e.g. "zemli-prototype (1).zip"
name = list(up)[0] if up else sorted(glob.glob("zemli-prototype*.zip"), key=os.path.getmtime)[-1]
subprocess.run(["unzip", "-q", "-o", name], check=True)
print("unzipped:", name)

## 2. Download the dataset and keep only cucumber leaves (about 1 GB download)

In [ ]:
import re, random, shutil
from pathlib import Path
from collections import Counter
from datasets import load_dataset

ds = load_dataset("Project-AgML/plant_leaf_disease_classification", split="train")
plant_names = ds.features["plant_type"].names
label_names = ds.features["label"].names
print("plant types:", plant_names)
cuc = plant_names.index(next(p for p in plant_names if "cucumber" in p.lower()))
cds = ds.filter(lambda p: p == cuc, input_columns="plant_type")
print("\ncucumber images per label:")
for k, v in Counter(cds["label"]).most_common():
    print(f"  {label_names[k]:30s} {v}")

## 3. Map dataset labels to Zemli class names

Labels are matched by keywords. Anything that doesn't match (fruit classes, other diseases) is skipped. Check the printout: every class you expect should appear.

In [ ]:
RULES = [  # (keyword regex, zemli class)
    (r"fresh|healthy|good", "healthy"),
    (r"down[e]?y", "downy_mildew"),
    (r"powdery", "powdery_mildew"),
    (r"anthracnose", "anthracnose"),
    (r"mosaic", "mosaic_virus"),
]
SKIP = r"augment|fruit|cucumber$|belly|pythium|rot"   # fruit images and augmented copies

def zemli_class(name):
    n = name.lower()
    for pat, cls in RULES:
        if re.search(pat, n):
            return cls
    return None

pool = {}   # zemli class -> list of (source, key)
for i, lab in enumerate(cds["label"]):
    cls = zemli_class(label_names[lab])
    if cls: pool.setdefault(cls, []).append(("hf", i))
for cls, items in pool.items():
    print(f"{cls:16s} {len(items)} images (Hugging Face)")

### (Optional) add another dataset
For example the Cucumber Disease Recognition Dataset from Mendeley (DOI 10.17632/y6d3z6f8z9.1). Download its zip in your browser, then run this cell and upload it. Only folders whose names match the keywords are used, and folders with "augment" in the path are skipped so the same leaf can't land in both train and test. Skip this cell if you don't have one.

In [ ]:
up = files.upload()
for name in up:
    shutil.unpack_archive(name, "extra")
    for p in Path("extra").rglob("*"):
        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
        path_l = str(p.parent).lower()
        if re.search(SKIP, path_l): continue
        cls = zemli_class(p.parent.name)
        if cls: pool.setdefault(cls, []).append(("file", str(p)))
for cls, items in pool.items():
    print(f"{cls:16s} {len(items)} images in total")

## 4. Split 80 / 10 / 10 and write image folders

In [ ]:
random.seed(42)
dst = Path("data/cucumber"); shutil.rmtree(dst, ignore_errors=True)
MIN_IMAGES = 60
for cls, items in sorted(pool.items()):
    if len(items) < MIN_IMAGES:
        print(f"skipping {cls}: only {len(items)} images"); continue
    random.shuffle(items)
    n = len(items); a, b = int(n * 0.8), int(n * 0.9)
    for split, part in [("train", items[:a]), ("val", items[a:b]), ("test_public", items[b:])]:
        d = dst / split / cls; d.mkdir(parents=True, exist_ok=True)
        for j, (src, key) in enumerate(part):
            out = d / f"{src}_{j}.jpg"
            if src == "hf": cds[key]["image"].convert("RGB").save(out, quality=92)
            else: shutil.copy(key, out)
    print(f"{cls:16s} {n:5d} -> train {a}, val {b-a}, test {n-b}")

## 5. Train (about 10-20 minutes)

In [ ]:
!python zemli/training/train.py --crop cucumber --data data/cucumber --epochs 12 --batch 64 --workers 2 --out out

## 6. Results

In [ ]:
import json
m = json.load(open("out/cucumber/metrics.json"))
print("classes:", m["classes"])
for split in ["val", "test_public", "test_field"]:
    if split in m: print(f"{split:12s} accuracy {m[split]['accuracy']:.4f}  macro-F1 {m[split]['macro_f1']:.4f}")
print()
for c, v in m["test_public"]["per_class"].items(): print(f"{c:16s} F1 {v['f1']:.3f}  ({v['support']} images)")

## 7. Download
Unzip into `backend\models\` so you get `backend\models\cucumber\model.onnx`, then restart uvicorn.

In [ ]:
!cd out && zip -q -r ../cucumber_model.zip cucumber/model.onnx cucumber/labels.txt cucumber/metrics.json
files.download("cucumber_model.zip")